# End-to-End Agentspan Notebook: Review a Service Design

This notebook is meant to explain what Agentspan actually does at runtime.

Agentspan is an execution layer for AI agents. You define agents and tools in Python, but the execution itself is managed by a server. That server stores execution state, history, and the data the UI needs to show what happened during a run.

The goal here is not to build the most sophisticated system-design agent possible. The goal is to show one compact end-to-end workflow that exercises the main pieces people need to understand:

It covers:
- an external integration
- custom Python tools
- multiple agents
- a built-in multi-agent strategy
- deploy, serve, and run
- monitoring the execution in the Agentspan UI

The scenario is intentionally small. We will review a proposed notification service and produce a short architecture recommendation. That gives the notebook a concrete job to do while keeping the focus on the Agentspan runtime model.

## 1. Install the tested release and choose an LLM provider

This notebook was developed and tested against `agentspan==0.1.8`.

If you are starting from a clean environment, install the tested release first. Then configure the environment variables for the model provider you want to use. The example below uses OpenAI because that is what was used for validation, but the notebook is not meant to require one specific provider. Replace those values with any supported provider configuration you prefer.

```bash
pip install agentspan==0.1.8 fastapi uvicorn requests
export OPENAI_API_KEY=your-own-key
export AGENTSPAN_LLM_MODEL=openai/gpt-4o-mini
```

If you are already inside Jupyter, the next cell installs the tested dependencies in the notebook kernel.

In [ ]:
%pip install -q agentspan==0.1.8 fastapi uvicorn requests


## 2. Load Python helpers and explain the local ports

Agentspan runs with a server and one or more Python workers. This notebook uses two local ports:

- `6773` for the Agentspan server
- `8001` for a tiny mock platform API that stands in for an external integration

The helper functions below do four jobs:
- run shell commands
- start background processes
- wait for HTTP endpoints to come up
- build links back into the Agentspan UI

In [ ]:
import atexit
import os
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

import requests
from IPython.display import HTML, display

DEMO_ROOT = Path.cwd()
SERVER_PORT = 6773
API_PORT = 8001
SERVER_ROOT = f"http://127.0.0.1:{SERVER_PORT}"
SERVER_URL = f"{SERVER_ROOT}/api"
API_ROOT = f"http://127.0.0.1:{API_PORT}"
DEFAULT_MODEL = os.environ.get("AGENTSPAN_LLM_MODEL", "openai/gpt-4o-mini")

processes = []


def run_shell(cmd: str, check: bool = True):
    """Run one shell command and print the result so notebook users can see the step."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return result


def wait_for_http(url: str, timeout_seconds: int = 60):
    """Poll one URL until it returns a successful response."""
    start = time.time()
    while time.time() - start < timeout_seconds:
        try:
            response = requests.get(url, timeout=2)
            if response.ok:
                return response
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"Timed out waiting for {url}")


def start_background(command: list[str], log_name: str):
    """Start a long-lived process and keep a handle so the notebook can stop it later."""
    log_path = DEMO_ROOT / log_name
    handle = open(log_path, "w")
    proc = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT)
    processes.append((proc, handle))
    return proc


def execution_link(execution_id: str) -> HTML:
    url = f"{SERVER_ROOT}/execution/{execution_id}"
    return HTML(f'<a href="{url}" target="_blank">Open execution {execution_id}</a>')


def executions_link() -> HTML:
    return HTML(f'<a href="{SERVER_ROOT}/executions" target="_blank">Open the execution list</a>')


def cleanup_processes():
    for proc, handle in reversed(processes):
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
        handle.close()


atexit.register(cleanup_processes)

print("agentspan version =", version("agentspan"))
print("model =", DEFAULT_MODEL)
print("server ui root =", SERVER_ROOT)


## 3. Write a small external API to integrate with

This cell writes a tiny FastAPI application to disk. The API publishes two endpoints:

- one endpoint that returns the service design brief
- one endpoint that returns the approved platform stack

Agentspan will call this API through `http_tool()` in a later step.

In [ ]:
%%writefile service_catalog_api.py
from fastapi import FastAPI

app = FastAPI(
    title="Platform Catalog API",
    version="1.0.0",
    description="A small mock API used by the Agentspan system design notebook.",
)

NOTIFICATION_SERVICE_BRIEF = {
    "service_name": "notification-service",
    "purpose": "Send email, SMS, and in-app notifications for customer events.",
    "peak_rps": 850,
    "avg_payload_kb": 6,
    "regions": ["us-east-1"],
    "contains_pii": True,
    "availability_target": "99.9%",
    "delivery_targets": ["email", "sms", "in_app"],
}

APPROVED_STACK = {
    "compute": ["kubernetes"],
    "queue": ["sqs", "kafka"],
    "primary_store": ["postgres", "dynamodb"],
    "cache": ["redis"],
    "secrets": ["aws-secrets-manager"],
    "observability": ["prometheus", "grafana", "structured-logs"],
}


@app.get("/services/notification-service")
def get_notification_service_brief():
    return NOTIFICATION_SERVICE_BRIEF


@app.get("/platform/approved-stack")
def get_approved_stack():
    return APPROVED_STACK


## 4. Start the mock API and the Agentspan server

The mock API is the external integration for the notebook.

The Agentspan server is the durable runtime. It stores execution state, history, and the data that the UI reads later.

In [ ]:
# Start the mock API if this notebook is the first process using the demo port.
try:
    wait_for_http(f"{API_ROOT}/openapi.json", timeout_seconds=2)
    print("mock api already running at", API_ROOT)
except Exception:
    start_background(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "service_catalog_api:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(API_PORT),
        ],
        "platform_api.log",
    )
    wait_for_http(f"{API_ROOT}/openapi.json")
    print("mock api ready at", API_ROOT)


In [ ]:
# Start the Agentspan server if it is not already listening on the demo port.
os.environ["AGENTSPAN_SERVER_URL"] = SERVER_URL

try:
    wait_for_http(f"{SERVER_ROOT}/actuator/health", timeout_seconds=2)
    print("agentspan server already running at", SERVER_ROOT)
except Exception:
    run_shell(f"agentspan server start --port {SERVER_PORT}")
    wait_for_http(f"{SERVER_ROOT}/actuator/health")
    print("agentspan server ready at", SERVER_ROOT)


## 5. Define the integration tools

In Agentspan, a *tool* is a callable capability an agent can use while it is working.

This notebook uses `http_tool()` to wrap two HTTP endpoints from the local API. These tools run on the server side, because the integration is a simple HTTP call and does not need a Python worker.

In [ ]:
from agentspan.agents import Agent, AgentRuntime, http_tool, scatter_gather, tool

# Each http_tool exposes one external API endpoint to the agent runtime.
get_notification_service_brief = http_tool(
    name="get_notification_service_brief",
    description="Read the design brief for the notification service.",
    url=f"{API_ROOT}/services/notification-service",
    method="GET",
)

get_approved_stack = http_tool(
    name="get_approved_stack",
    description="Read the approved platform stack for new services.",
    url=f"{API_ROOT}/platform/approved-stack",
    method="GET",
)


## 6. Define the custom Python tools

These tools use plain Python code. That is what `@tool` is for in Agentspan.

Unlike `http_tool()`, custom `@tool` functions run in a Python worker process. The runtime still records the calls on the server, but the function body runs in Python.

In [ ]:
STANDARD_TEXT = {
    "security": (
        "Services that handle PII should use a managed secret store, "
        "encrypt data in transit, and avoid storing provider credentials in code."
    ),
    "operations": (
        "New services should emit structured logs, export Prometheus metrics, "
        "and queue retries instead of retrying synchronously in request handlers."
    ),
    "capacity": (
        "Bursty workloads should decouple ingestion from delivery with a queue "
        "and keep worker scaling independent from API request handling."
    ),
}


@tool
def read_standard(section: str) -> str:
    """Read one architecture standard by section name."""
    return STANDARD_TEXT.get(section.lower(), "No standard found for that section.")


@tool
def estimate_capacity(peak_rps: int, avg_payload_kb: int) -> dict:
    """Estimate queue depth and worker count for a bursty service."""
    projected_messages_per_minute = peak_rps * 60
    projected_data_mb_per_minute = round((projected_messages_per_minute * avg_payload_kb) / 1024, 1)

    return {
        "projected_messages_per_minute": projected_messages_per_minute,
        "projected_data_mb_per_minute": projected_data_mb_per_minute,
        "recommended_worker_pool": 12 if peak_rps > 500 else 6,
        "recommended_queue": "sqs",
    }


## 7. Define the agents and the multi-agent strategy

This notebook uses two Agentspan concepts here:

- a worker agent that does one focused review task
- `scatter_gather()`, which is a built-in multi-agent strategy

`scatter_gather()` creates a coordinator agent. The coordinator breaks the request into parallel sub-tasks, runs the worker on each sub-task, and then synthesizes a final review summary.

In [ ]:
reviewer = Agent(
    name="service_reviewer",
    model=DEFAULT_MODEL,
    tools=[
        get_notification_service_brief,
        get_approved_stack,
        read_standard,
        estimate_capacity,
    ],
    instructions=(
        "You review one aspect of a proposed service design. "
        "Use the available tools to gather facts before you answer. "
        "Return concise bullet points and a short recommendation."
    ),
)

coordinator = scatter_gather(
    "system_design_review",
    reviewer,
    instructions=(
        "Create exactly three review tasks: security, operations, and capacity. "
        "Each task should inspect the service brief, the approved stack, and the relevant "
        "standard or estimate. After the three reviews finish, synthesize a final design review "
        "with these headings: Approved stack, Main risks, and Next steps."
    ),
)


## 8. Deploy and serve the workflow

`deploy()` registers the workflow definition on the Agentspan server.

`serve()` starts the Python worker process that can execute the custom `@tool` functions. The HTTP tools do not need the Python worker, but `read_standard()` and `estimate_capacity()` do.

In [ ]:
runtime = AgentRuntime(server_url=SERVER_URL)

# Register the workflow definition on the server.
runtime.deploy(coordinator)

# Start the Python worker in the background so the notebook can keep running.
runtime.serve(coordinator, blocking=False)

time.sleep(2)
print("workflow deployed and worker started")


## 9. Run one execution

The prompt below is the single request that drives the whole system.

This is the point where the runtime does the end-to-end work:
- start one execution
- fan out into parallel review tasks
- call the integration and Python tools
- gather the sub-results
- return a final review summary

In [ ]:
prompt = (
    "Review the notification service proposal and recommend an approved design. "
    "Make sure the final review names the queue, primary datastore, secrets manager, "
    "and observability stack."
)

result = runtime.run(coordinator, prompt)
review_text = result.output["result"] if isinstance(result.output, dict) else result.output

print("execution_id =", result.execution_id)
print("status =", result.status)
print("finish_reason =", result.finish_reason)
print()
print(review_text)


## 10. Monitor the run in the UI

Agentspan stores the execution on the server, so the UI can show what happened after the run finishes.

Open the execution detail page first. Then look at:
- the execution graph
- the task list
- the timing for each step

In [ ]:
display(executions_link())
display(execution_link(result.execution_id))


## 11. Optional cleanup

Run this cell when you are done inspecting the execution. It stops the worker processes that this notebook started. If you want to keep the UI open for a while, wait before running it.

In [ ]:
runtime.shutdown()
cleanup_processes()
print("notebook processes stopped")
